###Sai Kadiravan S
###Assignment -4

In [1]:
!pip install -U transformers datasets scikit-learn -q

In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

In [3]:
df = pd.read_csv('/content/IMDB Dataset.csv')

# Take only 10K samples (5K positive, 5K negative)
df = df.sample(n=10000, random_state=42).reset_index(drop=True)

df.head()

In [4]:
import re

def clean_text(text):
    text = re.sub(r'<.*?>', '', text)  # remove HTML
    text = re.sub(r'[^a-zA-Z ]', '', text)  # remove special chars
    text = text.lower()
    return text

df['review'] = df['review'].apply(clean_text)

# Convert labels to numbers
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

df.head()

In [5]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'], df['sentiment'], test_size=0.2, random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)

In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=128)

In [7]:
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDbDataset(train_encodings, train_labels)
val_dataset = IMDbDataset(val_encodings, val_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

In [9]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=4,   # 👈 reduced
    per_device_eval_batch_size=4,
    num_train_epochs=2
)

In [10]:
def compute_metrics(pred):
    logits, labels = pred
    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [12]:
trainer.train()

In [13]:
trainer.evaluate(test_dataset)

In [14]:
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = test_labels.values

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

In [19]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

for param in model.distilbert.parameters():
    param.requires_grad = False

# Create new trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [20]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# Freeze all first
for param in model.distilbert.parameters():
    param.requires_grad = False

# Unfreeze last 2 layers
for param in model.distilbert.transformer.layer[-2:].parameters():
    param.requires_grad = True

trainer = Trainer(model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics)

trainer.train()

In [22]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# Train everything
for param in model.parameters():
    param.requires_grad = True

trainer = Trainer(model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics)

trainer.train()

Comparison:

Experiment 1 (Frozen)        → Fastest, Lowest performance

Experiment 2 (Last 2 Layers) → Balanced performance

Experiment 3 (Full)          → Best performance, Slowest



A clear trade-off between performance and computational cost was observed:

Freezing layers resulted in faster training but lower accuracy

Partial fine-tuning provided a balance between efficiency and performance

Full fine-tuning achieved highest accuracy at the cost of increased training time

Although the assignment specifies the use of BERT, DistilBERT was used as an efficient alternative. DistilBERT is a distilled version of BERT that retains most of its language understanding capabilities while being significantly smaller and faster.

This choice was made due to computational constraints in the Colab environment, where full BERT training resulted in high memory usage and longer training times. DistilBERT enabled faster experimentation and efficient fine-tuning while maintaining strong performance.

Thus, the overall objective of understanding transformer-based fine-tuning and experimentation was successfully achieved.
